# Stage 3 Explainer Notebook (Interactive v2)

This notebook is for **tracing and understanding** model behavior, not only batch-running scripts.

Focus goals:
- inspect middle artifacts (data stats, coefficients, support vectors, train/test gaps)
- visualize model behavior (residuals, decision boundaries, elbow/silhouette)
- practice troubleshooting (scaling, overfit/leakage signals)
- connect local hardware (CPU/GPU) to training performance decisions


## Table Of Contents

1. Environment sync and preflight checks  
2. Linear regression tracing (with residual diagnosis)  
3. Logistic vs SVM decision boundaries (2D)  
4. Troubleshooting lab: cost of missing scaling  
5. KMeans: inertia vs silhouette analysis  
6. PyTorch CPU vs CUDA benchmark (with memory)  
7. Model leaderboard from Stage 3 artifacts  
8. Failure diagnosis review and leakage red-flag  
9. Optional script sync / fail-fast runner


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes, load_breast_cancer, load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    silhouette_score,
)

try:
    import torch
except Exception:
    torch = None

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda s: s

sns.set_theme(style='whitegrid')

STAGE = 3
STAGE_DIR = Path.cwd()
if not (STAGE_DIR / 'topic01_linear_regression.py').exists():
    # fallback when notebook opened from parent folder
    STAGE_DIR = Path.cwd() / 'src' / f'stage-{STAGE}'

RESULTS_DIR = STAGE_DIR / 'results'
STAGE_RESULTS_DIR = RESULTS_DIR / 'stage3'

print('Python:', sys.version.split()[0])
print('Stage dir:', STAGE_DIR)
print('Results dir:', RESULTS_DIR)


## 1) Environment Sync (Keep Fail-Fast Mindset)

This section keeps the original "verify first" workflow before interactive tracing.


In [ ]:
def run_cmd(cmd, cwd: Path | None = None):
    print('>>', ' '.join(map(str, cmd)))
    completed = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
    return completed.returncode

_ = run_cmd([sys.executable, '--version'])
_ = run_cmd([sys.executable, '-c', 'import sklearn, pandas, numpy; print(sklearn.__version__)'])
if torch is not None:
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
else:
    print('torch: not installed in current kernel')


## 2) Linear Regression Tracing (Middle Artifacts + Diagnosis)

We trace the lifecycle in cells: data -> summary -> fit -> metrics -> residual plot.


In [ ]:
# Data loading and method-chaining summary

data = load_diabetes(as_frame=True)
X = data.data
y = data.target

summary = (
    X.assign(target=y)
    .describe()
    .T[['mean', 'std', '50%']]
    .rename(columns={'50%': 'median'})
)

display(Markdown('### Feature Summary (Method Chaining)'))
display(summary)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lin_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', LinearRegression()),
])
lin_pipe.fit(X_train, y_train)

train_pred = lin_pipe.predict(X_train)
test_pred = lin_pipe.predict(X_test)

train_mse = mean_squared_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print(f'Train MSE: {train_mse:.3f}')
print(f'Test MSE : {test_mse:.3f}')
print(f'Test MAE : {test_mae:.3f}')
print(f'Test R2  : {test_r2:.3f}')

if test_r2 < 0.2:
    print('DIAGNOSIS: underfitting risk.')
elif (test_mse - train_mse) / max(train_mse, 1e-9) > 0.35:
    print('DIAGNOSIS: generalization gap risk.')
else:
    print('DIAGNOSIS: baseline is acceptable.')


In [ ]:
# Residual visualization (Pass-2 intuition)

residuals = y_test - test_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(test_pred, residuals, alpha=0.7)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Predicted')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')

sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.show()


### Reflection Prompt

- Do residuals look randomly spread around 0, or is there structure?  
- If structure exists, what feature transformation would you test first?


## 3) Logistic vs SVM Decision Boundary (2D Tracing)

We intentionally use 2 features (`mean radius`, `mean texture`) for visual intuition.


In [ ]:
bc = load_breast_cancer(as_frame=True)
feat_cols = ['mean radius', 'mean texture']
X2 = bc.data[feat_cols]
y2 = bc.target

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

logi = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=3000, random_state=42)),
])
svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', C=10, gamma=0.1, probability=True, random_state=42)),
])

logi.fit(X2_train, y2_train)
svm.fit(X2_train, y2_train)

for name, model in [('logistic', logi), ('svm_rbf', svm)]:
    pred = model.predict(X2_test)
    print(name, 'accuracy=', round(accuracy_score(y2_test, pred), 3), 'f1=', round(f1_score(y2_test, pred), 3))


In [ ]:
def plot_boundary(ax, model, X, y, title):
    x_min, x_max = X.iloc[:, 0].min() - 1.0, X.iloc[:, 0].max() + 1.0
    y_min, y_max = X.iloc[:, 1].min() - 1.0, X.iloc[:, 1].max() + 1.0
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 250),
        np.linspace(y_min, y_max, 250),
    )
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=X.columns)
    zz = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, zz, alpha=0.25, levels=2)
    ax.scatter(X.iloc[:, 0], X.iloc[:, 1], c=y, s=18, alpha=0.8)
    ax.set_xlabel(X.columns[0])
    ax.set_ylabel(X.columns[1])
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_boundary(axes[0], logi, X2_test, y2_test, 'Logistic Decision Boundary (2D)')
plot_boundary(axes[1], svm, X2_test, y2_test, 'SVM RBF Decision Boundary (2D)')
plt.tight_layout()
plt.show()


### Reflection Prompt

- Which boundary looks more flexible?  
- If flexibility increases but test F1 decreases, what failure pattern is likely?


### 3B) Interactive SVM Kernel Visualizer

Use sliders to see how `C` and `gamma` change the decision boundary.

- `C` controls penalty for misclassification (higher `C` can overfit).
- `gamma` controls boundary flexibility for RBF (higher `gamma` can create very wiggly boundaries).


In [ ]:
from sklearn.datasets import make_moons

# Synthetic non-linear dataset is ideal to visualize kernel effects.
Xm, ym = make_moons(n_samples=500, noise=0.25, random_state=42)


def _plot_svm_boundary(C=1.0, gamma=0.5, kernel='rbf'):
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=kernel, C=float(C), gamma=float(gamma), random_state=42)),
    ])
    model.fit(Xm, ym)

    xx, yy = np.meshgrid(
        np.linspace(Xm[:, 0].min() - 1.0, Xm[:, 0].max() + 1.0, 300),
        np.linspace(Xm[:, 1].min() - 1.0, Xm[:, 1].max() + 1.0, 300),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict(grid).reshape(xx.shape)

    pred = model.predict(Xm)
    acc = accuracy_score(ym, pred)
    n_sv = int(model.named_steps['svm'].n_support_.sum())

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, zz, alpha=0.25, levels=2)
    plt.scatter(Xm[:, 0], Xm[:, 1], c=ym, s=18, alpha=0.85)
    plt.title(f'SVM ({kernel}) | C={C:.3f}, gamma={gamma:.3f}, acc={acc:.3f}, support_vectors={n_sv}')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.tight_layout()
    plt.show()


try:
    import ipywidgets as widgets
    from ipywidgets import interact

    interact(
        _plot_svm_boundary,
        C=widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=2, step=0.1, description='C'),
        gamma=widgets.FloatLogSlider(value=0.5, base=10, min=-3, max=1, step=0.1, description='gamma'),
        kernel=widgets.Dropdown(options=['rbf', 'poly', 'linear'], value='rbf', description='kernel'),
    )
except Exception:
    print('ipywidgets not available. Showing fallback static comparisons.')
    for C, gamma in [(0.3, 0.5), (1.0, 0.5), (10.0, 0.1), (10.0, 1.0)]:
        _plot_svm_boundary(C=C, gamma=gamma, kernel='rbf')


### Reflection Prompt

- Which setting gives smooth generalization vs. overly fragmented boundaries?  
- If support vector count is very high and boundary is wiggly, what does that imply about overfitting risk?


## 4) Troubleshooting Lab: The Cost of Missing Scaling (SVM)

This is a live "what-if" for issue identification.


In [ ]:
wine = load_wine(as_frame=True)
Xw = wine.data
yw = wine.target

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.25, random_state=42, stratify=yw
)

svm_no_scale = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_with_scale = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)),
])

svm_no_scale.fit(Xw_train, yw_train)
svm_with_scale.fit(Xw_train, yw_train)

pred_no = svm_no_scale.predict(Xw_test)
pred_yes = svm_with_scale.predict(Xw_test)

acc_no = accuracy_score(yw_test, pred_no)
acc_yes = accuracy_score(yw_test, pred_yes)

sv_count_no = int(svm_no_scale.n_support_.sum())
sv_count_yes = int(svm_with_scale.named_steps['svm'].n_support_.sum())

print(f'Without scaling: accuracy={acc_no:.3f}, support_vectors={sv_count_no}')
print(f'With scaling   : accuracy={acc_yes:.3f}, support_vectors={sv_count_yes}')

if acc_yes - acc_no > 0.05:
    print('DIAGNOSIS: missing scaling was a major root cause.')
else:
    print('DIAGNOSIS: scaling impact is small here; inspect C/gamma search next.')


### Reflection Prompt

- If support vector count increases significantly, what does that suggest about margin complexity?  
- Which would you tune first after scaling: `C` or `gamma`?


## 5) KMeans Intuition: Inertia vs Silhouette

Compare compactness (`inertia`) with separation (`silhouette`) to avoid one-metric mistakes.


In [ ]:
iris = load_iris(as_frame=True)
Xi = iris.data

scaler = StandardScaler()
Xi_scaled = scaler.fit_transform(Xi)

rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(Xi_scaled)
    rows.append({
        'k': k,
        'inertia': float(km.inertia_),
        'silhouette': float(silhouette_score(Xi_scaled, labels)),
    })

k_df = pd.DataFrame(rows)
display(k_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(k_df['k'], k_df['inertia'], marker='o')
axes[0].set_title('Elbow: Inertia vs K')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')

axes[1].plot(k_df['k'], k_df['silhouette'], marker='o', color='green')
axes[1].set_title('Silhouette vs K')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette')
plt.tight_layout()
plt.show()


### Reflection Prompt

- Does the K minimizing inertia always maximize silhouette?  
- Which K would you choose for interpretability vs segmentation quality?


## 6) Sovereign AI Hardware Bridge: CPU vs CUDA Benchmark

This cell compares local matrix multiplication on CPU vs GPU and shows peak VRAM usage.


In [ ]:
if torch is None:
    print('torch is not available in this notebook kernel.')
else:
    size = 4096  # raise to 10000 if your machine has enough RAM/VRAM
    a = torch.randn(size, size)
    b = torch.randn(size, size)

    t0 = time.time()
    _ = torch.matmul(a, b)
    cpu_time = time.time() - t0
    print(f'CPU matmul time: {cpu_time:.4f}s')

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        t_transfer_start = time.time()
        a_gpu = a.cuda()
        b_gpu = b.cuda()
        torch.cuda.synchronize()
        transfer_time = time.time() - t_transfer_start

        t1 = time.time()
        _ = torch.matmul(a_gpu, b_gpu)
        torch.cuda.synchronize()
        gpu_compute_time = time.time() - t1

        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        total_gpu = transfer_time + gpu_compute_time

        print(f'GPU transfer time: {transfer_time:.4f}s')
        print(f'GPU compute time : {gpu_compute_time:.4f}s')
        print(f'GPU total time   : {total_gpu:.4f}s')
        print(f'Peak GPU memory  : {peak_mb:.2f} MB')

        if total_gpu < cpu_time:
            print('DECISION: GPU path wins for this workload.')
        else:
            print('DECISION: CPU path is faster for this workload (overhead dominates).')
    else:
        print('CUDA unavailable in current kernel; GPU benchmark skipped.')


## 7) Final Comparison Report (Modern Pandas Styling)

Load generated artifacts and render actionable leaderboard views.


In [ ]:
leaderboard_path = RESULTS_DIR / 'topic07_fair_comparison.csv'
if leaderboard_path.exists():
    df = pd.read_csv(leaderboard_path)
    display(Markdown('### One-Split Leaderboard'))
    leaderboard = (
        df.sort_values('test_f1', ascending=False)
        .assign(stability=lambda d: d['generalization_gap_f1'].abs().apply(lambda g: 'stable' if g < 0.03 else 'watch'))
    )
    styled = (
        leaderboard.style
        .format(precision=3)
        .background_gradient(subset=['test_f1', 'test_accuracy'], cmap='RdYlGn')
        .background_gradient(subset=['generalization_gap_f1'], cmap='YlOrRd')
    )
    display(styled)
else:
    print('Missing:', leaderboard_path)

seed_path = RESULTS_DIR / 'topic07_fair_comparison_seed_stats.csv'
if seed_path.exists():
    seed_df = pd.read_csv(seed_path).sort_values('test_f1_mean', ascending=False)
    display(Markdown('### Multi-Seed Stability (Mean/Std)'))
    display(seed_df.style.format(precision=3).background_gradient(subset=['test_f1_mean'], cmap='RdYlGn'))
else:
    print('Missing:', seed_path)


### Reflection Prompt

- If the top one-split model has higher `test_f1` but worse `test_f1_std`, would you still choose it for production?  
- Which model is the most stable candidate from the seed table?


## 8) Failure Diagnosis Review + Leakage Red-Flag

Use generated failure artifacts and a quick leakage-correlation demonstration.


In [ ]:
diag_path = STAGE_RESULTS_DIR / 'failure_diagnosis.md'
if diag_path.exists():
    display(Markdown(diag_path.read_text(encoding='utf-8')))
else:
    print('Missing:', diag_path)

# Small interactive leakage demonstration:
# add a leaky feature that directly encodes target with tiny noise.
rng = np.random.default_rng(42)
N = 600
y_leak = rng.integers(0, 2, size=N)
X_noise = rng.normal(size=(N, 5))
leaky_col = y_leak + 0.01 * rng.normal(size=N)

leak_df = pd.DataFrame(X_noise, columns=[f'noise_{i}' for i in range(5)])
leak_df['leaky_feature'] = leaky_col
leak_df['target'] = y_leak

corr = leak_df.corr(numeric_only=True)['target'].drop('target').sort_values(key=np.abs, ascending=False)
print('Top correlations with target (absolute):')
print(corr)

plt.figure(figsize=(7, 4))
sns.barplot(x=corr.index, y=corr.values)
plt.title('Leakage Red-Flag: Feature Correlation With Target')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Rule: if a feature correlates suspiciously with target, audit for leakage immediately.')


## 9) Optional Script Sync (Batch Mode)

You can still run scripts end-to-end from this notebook after interactive tracing.


In [ ]:
RUN_BATCH = False  # set True when you want full script execution

if RUN_BATCH:
    scripts = [
        'topic01_linear_regression.py',
        'topic02_logistic_regression.py',
        'topic03_decision_tree_depth.py',
        'topic04_random_forest_baseline.py',
        'topic05_svm_tuning.py',
        'topic06_kmeans_silhouette.py',
        'topic07_fair_model_comparison.py',
        'topic08_failure_modes_overfit_leakage.py',
    ]

    for s in scripts:
        rc = run_cmd([sys.executable, str(STAGE_DIR / s)], cwd=STAGE_DIR)
        if rc != 0:
            raise RuntimeError(f'Failed script: {s}')

    # Include GPU bridge when available.
    if torch is not None:
        _ = run_cmd([sys.executable, str(STAGE_DIR / 'topic09_pytorch_cuda_bridge.py')], cwd=STAGE_DIR)

    # Optional fail-fast runner.
    _ = run_cmd([
        'powershell', '-ExecutionPolicy', 'Bypass', '-File', str(STAGE_DIR / 'run_all_stage3.ps1')
    ], cwd=STAGE_DIR)
else:
    print('Batch mode skipped. Set RUN_BATCH=True to execute all stage scripts.')


## Study Checklist Before Stage 4

- I can explain at least one failure diagnosis from evidence, not intuition.  
- I can justify model choice using both performance and stability.  
- I can show when GPU helps and when transfer overhead makes CPU better.  
- I can reproduce Stage 3 artifacts (`results/stage3/*`) from a clean run.
